In [95]:
from gitsource import chunk_documents, GithubRepositoryDataReader
from minsearch import Index
from rich.pretty import pprint
from typing import List, Dict, Any

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner



# Question 1

In [77]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [78]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [79]:
print(len(documents))

72


# Question 2

In [80]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [81]:
INSTRUCTIONS = '''
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

class RAGBase:                 
    def __init__(self,
                 index,
                 llm_client,
                 instructions=INSTRUCTIONS,
                 prompt_template=USER_PROMPT_TEMPLATE,
                 model='gpt-5.4-mini'
                 ):
                    self.index = index
                    self.llm_client = llm_client
                    self.instructions = instructions
                    self.prompt_template = prompt_template
                    self.model = model
                    # ,

    def search(self, question):
        
        return self.index.search(
            question,
            num_results=5
        )
        
    def build_context(self, search_results):
        lines = []
        for doc in search_results: 
            lines.append(doc["content"])
            lines.append(doc["filename"])
            lines.append("") 
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        
        context = self.build_context(search_results)
        prompt = self.prompt_template.format(question=query, context=context)
        return prompt.strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


In [82]:
llm_client = OpenAI()

In [83]:
index = build_index(documents)

In [84]:
rag_instance = RAGBase(index, llm_client)

In [85]:
pprint(rag_instance.search("How does the agentic loop keep calling the model until it stops?"))

[
│   {
│   │   'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry out the task. For\n  us that\'s only `search`.\n- Memory, the message history. We append every prompt, every model\n  output, and every tool result. The agent reads this to know what it\n  has already tried.\n\nEverything below is the code that wires these three together inside a\nloop.\n\n## A developer prompt\n\nSo far we\'ve relied on the model to figure out when to search. We make\nthat more reliable with a `developer` message that spells out how to\nbehave. This is where we give the agent its role. The same message\nalso pushes it toward multiple searches, so we get to watch the loop\nrun more than once.\n\n```python\ninstructions = """\nYou\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n""".strip()\n```\n\n## A function-call helper\n\nWe\'ll be running function calls repeatedly inside the loop, so let\'s\nwrap that in a small helper. It turns the JSON arguments into a Python\ndict, calls the right function, and serializes the result. We only\nhave one tool for now, so we dispatch on the function name directly.\n\n```python\ndef make_call(call):\n    args = json.loads(call.arguments)\n\n    if call.name == "search":\n        result = search(**args)\n\n    result_json = json.dumps(result, indent=2)\n\n    return {\n        "type": "function_call_output",\n        "call_id": call.call_id,\n        "output": result_json,\n    }\n```\n\nThe helper returns the exact structure the Responses API expects.\nWhen we add more tools later, we\'ll extend this with more `if`\nbranches (or switch to a registry).\n\n## Processing one response\n\nLet\'s process a single model response. We append each output entry to\nthe conversation, print any messages, and run any function calls.\nFunction-call results get appended too.\n\n```python\nquestion = "I just discovered the course. Can I join it?"\n\nmessages = [\n    {"role": "developer", "content": instructions},\n    {"role": "user", "content": question},\n]\n\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nmessages.extend(response.output)\nhas_function_calls = False\n\nfor item in response.output:\n    if item.type == "function_call":\n        print("function_call:", item.name, item.arguments)\n        call_output = make_call(item)\n        messages.append(call_output)\n        has_function_calls = True\n\n    elif item.type == "message":\n        print("ASSISTANT:")\n        print(item.content[0].text)\n```\n\nThe `has_function_calls` flag tells us whether the model needs another\nAPI call. If the response con

# Question 3

In [87]:
answer = rag_instance.rag("How does the agentic loop keep calling the model until it stops?")

pprint(answer.usage.input_tokens)

7092

# Question 4

In [88]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


# Question 5

In [91]:
index_with_chunks = build_index(chunks)
rag_instance_with_chunks = RAGBase(index_with_chunks, llm_client)

In [92]:
answer_q5 = rag_instance_with_chunks.rag("How does the agentic loop keep calling the model until it stops?")

pprint(answer.usage.input_tokens)

7092

# Question 6

In [104]:
def make_search_tool(index):
    """Factory that binds a specific index to a search function, avoiding global-state lookup."""
    def search(query: str) -> List[Dict[str, Any]]:
        """
        Search the course chunk index for content relevant to a query.

        Use this to look up specific topics, terms, or concepts from the
        course material. Call it multiple times with different keyword
        phrasings if the first search doesn't surface what you need.

        Args:
            query (str): Search keywords or a natural-language question.

        Returns:
            List[Dict[str, Any]]: Up to 5 matching chunks.
        """
        return index.search(query, num_results=5)
    return search

# explicit, unambiguous — this search function is permanently bound to chunk_index
chunk_index = build_index(chunks)
search = make_search_tool(chunk_index)

tools = Tools()
tools.add_tool(search)

In [105]:
developer_prompt = """
You're a course teaching assistant. Answer the student's question using
the search tool. Make multiple searches with different keywords before
answering.
"""

In [106]:
chat_interface = IPythonChatInterface()
llm_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI(),
)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

In [107]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)

print(result.last_message)

The **agentic loop** is a repeated cycle where the model can decide to use tools before giving the final answer.

A simple version looks like this:

1. Send the user prompt to the LLM
2. The LLM may return a tool call instead of a final answer
3. Your code executes the tool
4. You send the tool result back to the LLM
5. Repeat until the model returns a final answer with no more tool calls

In the course, this was described as a `while True` loop that:
- calls the LLM
- executes any function/tool calls it requests
- appends tool outputs to the conversation history
- stops when the model is done

So the key idea is: **the model gets to decide whether it needs another step**.

## How that differs from plain RAG

**Plain RAG** is usually a fixed pipeline:

```python
question -> retrieve docs -> build prompt -> answer
```

In other words, it’s typically:
- **one retrieval step**
- **one LLM answer step**
- no iterative decision-making

The course explicitly contrasts this with agents by say

In [ ]:
search_calls = [
    msg for msg in result.all_messages
    if getattr(msg, "type", None) == "function_call" and getattr(msg, "name", None) == "search"
]
answer_q6 = len(search_calls)

search was called 3 times


In [ ]:
print(f"search was called {answer_q6} times")